<a href="https://colab.research.google.com/github/Vitorms085/atividades-resolvidas-gsi073/blob/main/aula0_3_redes_neurais_alterado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [ ]:
import torch; import sklearn; from torch import nn

# 1. Carregar dados
iris = sklearn.datasets.load_iris()
X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 2. Preparar dados para pytorch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

## Definição do modelo e treinamento

In [ ]:
# 3. Definir modelo
import torch.nn.functional as F
class RedeNeural(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(RedeNeural, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Criar modelo
modelo = RedeNeural(4, 8, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

import copy
modelo_clonado = copy.deepcopy(modelo)

learning_rate = 0.1

# Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=learning_rate)

## Execução do treinamento com optimizer SGD

In [ ]:
# Loop de treino
for epoch in range(1000):
    optimizer.zero_grad()           # Limpa gradientes
    outputs = modelo(X)             # Forward
    loss = funcao_perda(outputs, y) # Calcula perda
    loss.backward()                 # Calcula derivadas do gradiente
    optimizer.step()                # Aplica regra de alteração dos parâmetros

    if (epoch + 1) % 100 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

# Treino com regra de gradiente descendente manual

In [ ]:
for epoch in range(1000):
    optimizer.zero_grad()           # Limpa gradientes
    outputs = modelo_clonado(X)     # Forward
    loss = funcao_perda(outputs, y) # Calcula perda
    loss.backward()                 # Calcula derivadas do gradiente

    with torch.no_grad():
        for param in modelo_clonado.parameters():
            param -= learning_rate * param.grad  # regra de atualização de pesos

    if (epoch + 1) % 100 == 0:
        print(f"Época [{epoch+1}/1000], Loss: {loss.item():.4f}")

## Avaliação do modelo

In [ ]:

# Avaliação do modelo treinado com SGD
with torch.no_grad():
    logits = modelo(X)
    probabilidades = torch.sigmoid(logits)
    predicoes = (probabilidades >= 0.5).float()

acuracia = (predicoes == y).float().mean().item()
print(f"Acurácia do modelo (SGD): {acuracia * 100:.2f}%")

# Relatório detalhado
from sklearn.metrics import classification_report
print("\nRelatório de classificação:")
print(classification_report(
    y.numpy().astype(int),
    predicoes.numpy().astype(int),
    target_names=["Não-Versicolor", "Versicolor"]
))


## Efeito do número de neurônios na camada escondida (hidden_dim) sobre a loss na época 100

In [ ]:

import matplotlib.pyplot as plt

hidden_dims = [1, 2, 4, 8, 16, 32, 64, 128]
losses_epoch100 = []

for hidden_dim in hidden_dims:
    torch.manual_seed(42)
    modelo_exp = RedeNeural(4, hidden_dim, 1)
    optimizer_exp = torch.optim.SGD(modelo_exp.parameters(), lr=0.1)
    loss_epoch100 = None

    for epoch in range(100):
        optimizer_exp.zero_grad()
        outputs = modelo_exp(X)
        loss = funcao_perda(outputs, y)
        loss.backward()
        optimizer_exp.step()

    loss_epoch100 = loss.item()
    losses_epoch100.append(loss_epoch100)
    print(f"hidden_dim={hidden_dim:4d} → Loss na época 100: {loss_epoch100:.4f}")

# Gráfico
plt.figure(figsize=(8, 4))
plt.plot(hidden_dims, losses_epoch100, marker='o', color='steelblue')
plt.xscale('log', base=2)
plt.xlabel("Número de neurônios na camada escondida (hidden_dim)")
plt.ylabel("Loss na época 100 (BCE)")
plt.title("Redução da loss conforme aumento de hidden_dim")
plt.grid(True)
plt.xticks(hidden_dims, labels=[str(h) for h in hidden_dims])
plt.tight_layout()
plt.show()
